# Ligand preparation

Code to prepare ligands of interest for the RASSCoL pipeline.

## Directory creation

In [4]:
# builtins
from pathlib import Path
import subprocess

# local
from rasscol_src.general_utils import *
from rasscol_src.rasscol_utils import *

ligand_name = 'sterone'
ligand_short_name = 'STR'
smiles = 'C1CCC2C(C1)CCC3C2CCC4C3CCC4'


## Ligand PDBQT creation

In [5]:
# usually takes several minutes
    
date, now = get_timestamp()

data_dir = Path('../data')
ligand_smi_dir = data_dir / f'{date}/00_ligand/'
ligand_smi_dir.mkdir(parents=True, exist_ok=True)

ligand_smi_path = ligand_smi_dir / f'{ligand_name}.smi'
ligand_pdbqt_path = ligand_smi_path.with_suffix('.pdbqt')

with ligand_smi_path.open('w') as f:
    f.write(smiles)


# obabel will try to minimise the 3d structure. This option takes time. 
# there are different speeds (number of cycles) you can chose from.
# --------------------------------------------------------------------------------------------
# option	    description
# --------------------------------------------------------------------------------------------
# fastest	    No cleanup
# fast	        Force field cleanup (100 cycles)
# med (default)	Force field cleanup (100 cycles) + Fast rotor search (only one permutation)
# slow	        Force field cleanup (250 cycles) + Fast rotor search (permute central rotors)
# slowest	    Force field cleanup (500 cycles) + Slow rotor search
# better	    Same as slow
# best	        Same as slowest
# dist, dg	    Use distance geometry method (unstable)

cleanup_speed = 'med'

obabel_path = Path('/usr/bin/obabel')
obabel_log_path = ligand_smi_dir.parent / f'{now}_run_obabel.log'

obabel_cmd = [
    obabel_path, 
    '-ismi', ligand_smi_path,
    '-opdbqt', '-O', ligand_pdbqt_path,
    '--gen3d', cleanup_speed,
    '-p', '7.4'
]

with obabel_log_path.open('w') as f:
    subprocess.run(obabel_cmd, stdout=f, stderr=f)

tidy_ligand_pdbqt(ligand_pdbqt_path, ligand_name, ligand_short_name)


In [ ]:
from rdkit import Chem
import py3Dmol

# Load the SDF file
mol = Chem.MolFromPDBFile(str(ligand_pdbqt_path))

mol_length, start_point, end_point = get_mol_len(get_pdbqt_coords(ligand_pdbqt_path))

# Visualize the molecule and draw the principal axis using Py3Dmol
viewer = py3Dmol.view(width=400, height=400)

# Add the molecule
block = Chem.MolToMolBlock(mol)
viewer.addModel(block, "mol")

# Add principal axis as an arrow spanning between the furthest points
viewer.addArrow({
    'start': {'x': start_point[0], 'y': start_point[1], 'z': start_point[2]},
    'end': {'x': end_point[0], 'y': end_point[1], 'z': end_point[2]},
    'color': 'red',
    'radius': 0.3
})

# Adjust view
viewer.setStyle({'stick': {}})
viewer.zoomTo()
viewer.show()

# Ensure hydrogens are explicitly included
mol = Chem.AddHs(mol)

# Get atom counts and types
atom_counts = {}
for atom in mol.GetAtoms():
    atom_type = atom.GetSymbol()
    if atom_type in atom_counts:
        atom_counts[atom_type] += 1
    else:
        atom_counts[atom_type] = 1

# Get number of rings and aromatic rings
num_rings = len(Chem.GetSymmSSSR(mol))
num_aromatic_rings = sum(1 for ring in Chem.GetSymmSSSR(mol) if all(mol.GetAtomWithIdx(idx).GetIsAromatic() for idx in ring))

# Print results
print(F"Atom counts: {atom_counts}")
print("Number of rings:", num_rings)
print("Number of aromatic rings:", num_aromatic_rings)
print("Bondi volume: ", calc_bondi_vol(atom_counts, num_rings, num_aromatic_rings), "Å3")
print(f"Length: {mol_length:.1f} Å")
print("\n---")
